# 📘 Session 10: Pandas Filtering, Sorting, and Advanced Data Operations
### Duration: ~2 Hours

---

**Topics Covered:**
1. Advanced Filtering Patterns
2. Sorting, Ranking, and Selecting Top/Bottom Rows
3. Conditional Column Creation & Replacement
4. Window Functions and Rolling Calculations
5. Time Series Basics with DateIndex and Resampling

---

**Why This Session Matters for Data Science:**
- Efficient filtering and sorting unlock faster analysis of large tables
- Conditional transformations power feature engineering and data cleaning
- Window and rolling functions support trend analysis and time-based features
- Time series resampling is essential for business reporting and forecasting

> This session continues the pandas journey by teaching the deeper tools used in real data pipelines and analytics workflows.

---
## 1. Advanced Filtering Patterns

Filtering is more than just `df[df['col'] > 10]`. Advanced patterns help you answer business questions cleanly.

### Common Filtering Methods

| Method | Purpose | Example |
|--------|---------|---------|
| `.query()` | Readable boolean logic | `df.query('Age > 30 and Department == 
')` |
| `.isin()` | Membership test | `df[df['State'].isin(['CA', 'TX'])]` |
| `.between()` | Range inclusion | `df[df['Date'].between('2024-01-01', '2024-03-31')]` |
| `.str.contains()` | Text matching | `df[df['Title'].str.contains('Manager', case=False)]` |
| `.duplicated()` | Find duplicates | `df[df.duplicated(['name', 'email'], keep=False)]` |
| `.loc[]` with boolean mask | Flexible row/column selection | `df.loc[mask, ['A', 'B']]` |

> ⚠️ **Special Case — `.query()` and variable names**: Use backticks for columns with spaces or invalid Python names, e.g. `df.query('`Sales Amount` > 1000')`.

In [2]:
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    'OrderID': [101, 102, 103, 104, 105, 106, 107, 108],
    'Customer': ['Alice', 'Bob', 'Charlie', 'Alice', 'Eve', 'Frank', 'Bob', 'Charlie'],
    'State': ['CA', 'NY', 'CA', 'TX', 'TX', 'NY', 'CA', 'TX'],
    'Product': ['Laptop', 'Phone', 'Laptop', 'Tablet', 'Phone', 'Laptop', 'Tablet', 'Phone'],
    'Quantity': [1, 2, 1, 3, 2, 4, 1, 2],
    'Sales': [1200, 800, 1250, 900, 820, 1300, 950, 780],
    'Order Date': pd.to_datetime(['2024-01-10', '2024-01-15', '2024-02-01', '2024-02-18', '2024-03-05', '2024-03-10', '2024-03-12', '2024-04-01'])
})

print('Sales data:')
print(sales)
print()

# Query with multiple conditions
high_value = sales.query('Sales > 900 and State == "CA"')
print('High-value California orders:')
print(high_value)
print()

# Membership test with isin
west_sales = sales[sales['State'].isin(['CA', 'TX'])]
print('West coast sales orders:')
print(west_sales)
print()

# Range filter with between
march_sales = sales[sales['Order Date'].between('2024-03-01', '2024-03-31')]
print('March sales orders:')
print(march_sales)
print()

# Text filtering with str.contains
phone_orders = sales[sales['Product'].str.contains('Phone', case=False)]
print('Phone orders:')
print(phone_orders)
print()

# Find duplicate customers and products
duplicate_orders = sales[sales.duplicated(['Customer', 'Product'], keep=False)]
print('Duplicate customer-product combinations:')
print(duplicate_orders)

Sales data:
   OrderID Customer State Product  Quantity  Sales Order Date
0      101    Alice    CA  Laptop         1   1200 2024-01-10
1      102      Bob    NY   Phone         2    800 2024-01-15
2      103  Charlie    CA  Laptop         1   1250 2024-02-01
3      104    Alice    TX  Tablet         3    900 2024-02-18
4      105      Eve    TX   Phone         2    820 2024-03-05
5      106    Frank    NY  Laptop         4   1300 2024-03-10
6      107      Bob    CA  Tablet         1    950 2024-03-12
7      108  Charlie    TX   Phone         2    780 2024-04-01

High-value California orders:
   OrderID Customer State Product  Quantity  Sales Order Date
0      101    Alice    CA  Laptop         1   1200 2024-01-10
2      103  Charlie    CA  Laptop         1   1250 2024-02-01
6      107      Bob    CA  Tablet         1    950 2024-03-12

West coast sales orders:
   OrderID Customer State Product  Quantity  Sales Order Date
0      101    Alice    CA  Laptop         1   1200 2024-01-10
2

---
## 2. Sorting, Ranking, and Selecting Top/Bottom Rows

Sorting and ranking are essential for identifying leaders, outliers, and trends.

### Sorting Methods

| Method | Use Case | Example |
|--------|----------|---------|
| `.sort_values()` | Sort rows by column values | `df.sort_values('Sales', ascending=False)` |
| `.sort_index()` | Sort index labels | `df.sort_index()` |
| `.nlargest()` / `.nsmallest()` | Top/bottom n rows | `df.nlargest(3, 'Sales')` |
| `.rank()` | Rank values within a column | `df['Rank'] = df['Sales'].rank(ascending=False)` |

> ⚠️ **Special Case — sort_values with missing values**: Use `na_position='first'` or `na_position='last'` to control where NaNs appear.

In [3]:
# --- Sorting and ranking examples ---
sales = sales.copy()
sales['Profit'] = [300, 120, 320, 180, 140, 360, np.nan, 110]  # include missing value for demonstration

print('Sorted by Sales descending:')
print(sales.sort_values('Sales', ascending=False))
print()

print('Top 3 orders by Profit:')
print(sales.nlargest(3, 'Profit'))
print()

print('Bottom 2 orders by Profit:')
print(sales.nsmallest(2, 'Profit'))
print()

sales['Profit Rank'] = sales['Profit'].rank(ascending=False, method='min')
print('Profit rank (higher profit = rank 1):')
print(sales[['OrderID', 'Profit', 'Profit Rank']])
print()

print('Sort by State then by Sales:')
print(sales.sort_values(['State', 'Sales'], ascending=[True, False]))
print()

print('Sort values with missing Profit last:')
print(sales.sort_values('Profit', na_position='last'))

Sorted by Sales descending:
   OrderID Customer State Product  Quantity  Sales Order Date  Profit
5      106    Frank    NY  Laptop         4   1300 2024-03-10   360.0
2      103  Charlie    CA  Laptop         1   1250 2024-02-01   320.0
0      101    Alice    CA  Laptop         1   1200 2024-01-10   300.0
6      107      Bob    CA  Tablet         1    950 2024-03-12     NaN
3      104    Alice    TX  Tablet         3    900 2024-02-18   180.0
4      105      Eve    TX   Phone         2    820 2024-03-05   140.0
1      102      Bob    NY   Phone         2    800 2024-01-15   120.0
7      108  Charlie    TX   Phone         2    780 2024-04-01   110.0

Top 3 orders by Profit:
   OrderID Customer State Product  Quantity  Sales Order Date  Profit
5      106    Frank    NY  Laptop         4   1300 2024-03-10   360.0
2      103  Charlie    CA  Laptop         1   1250 2024-02-01   320.0
0      101    Alice    CA  Laptop         1   1200 2024-01-10   300.0

Bottom 2 orders by Profit:
   OrderI

---
## 3. Conditional Column Creation & Replacement

Conditionally creating or replacing values is a core feature of feature engineering.

### Conditional Methods

| Method | Purpose | Example |
|--------|---------|---------|
| `np.where()` | Vectorized if/else | `np.where(df['Sales'] > 1000, 'High', 'Low')` |
| `.mask()` | Replace where condition is True | `df['col'].mask(cond, other)` |
| `.where()` | Keep where condition is True | `df['col'].where(cond, other)` |
| `.clip()` | Cap values with bounds | `df['Sales'].clip(lower=0, upper=2000)` |
| `.replace()` | Map or replace values | `df['Region'].replace({'NY': 'New York'})` |

> ⚠️ **Special Case — `np.where` returns arrays**: Assign the result directly to a Series or DataFrame column.

In [4]:
# --- Conditional transformations ---
sales = sales.copy()

sales['Sales Category'] = np.where(sales['Sales'] >= 1000, 'High', 'Medium')
sales['Profit Status'] = np.where(sales['Profit'] > 200, 'Good', 'Needs Review')

print('Sales with conditional categories:')
print(sales[['OrderID', 'Sales', 'Sales Category', 'Profit', 'Profit Status']])
print()

# Using where and mask
sales['Profit Capped'] = sales['Profit'].where(sales['Profit'] <= 300, 300)
sales['Profit Cleaned'] = sales['Profit'].mask(sales['Profit'].isna(), 0)
print('Profit capped and cleaned:')
print(sales[['OrderID', 'Profit', 'Profit Capped', 'Profit Cleaned']])
print()

# Using clip
sales['Quantity Clipped'] = sales['Quantity'].clip(lower=1, upper=3)
print('Quantity clipped to range 1-3:')
print(sales[['OrderID', 'Quantity', 'Quantity Clipped']])
print()

# Replace values in a text column
sales['State Clean'] = sales['State'].replace({'NY': 'New York', 'CA': 'California'})
print('State replacement mapping:')
print(sales[['State', 'State Clean']])

Sales with conditional categories:
   OrderID  Sales Sales Category  Profit Profit Status
0      101   1200           High   300.0          Good
1      102    800         Medium   120.0  Needs Review
2      103   1250           High   320.0          Good
3      104    900         Medium   180.0  Needs Review
4      105    820         Medium   140.0  Needs Review
5      106   1300           High   360.0          Good
6      107    950         Medium     NaN  Needs Review
7      108    780         Medium   110.0  Needs Review

Profit capped and cleaned:
   OrderID  Profit  Profit Capped  Profit Cleaned
0      101   300.0          300.0           300.0
1      102   120.0          120.0           120.0
2      103   320.0          300.0           320.0
3      104   180.0          180.0           180.0
4      105   140.0          140.0           140.0
5      106   360.0          300.0           360.0
6      107     NaN          300.0             0.0
7      108   110.0          110.0         

---
## 4. Window Functions and Rolling Calculations

Window functions let you calculate metrics over a moving window or group. They are key for trend detection and derived features.

### Window Methods

| Method | Purpose | Example |
|--------|---------|---------|
| `.rolling(window)` | Moving window statistics | `df['Sales'].rolling(3).mean()` |
| `.expanding()` | Cumulative calculations | `df['Sales'].expanding().sum()` |
| `.shift()` | Lag / lead values | `df['Sales'].shift(1)` |
| `.pct_change()` | Percent change | `df['Sales'].pct_change()` |
| `.diff()` | Difference between rows | `df['Sales'].diff()` |

> **Data Science relevance**: Rolling features capture momentum, seasonality, and rate-of-change for predictive models.

In [5]:
# --- Rolling and window calculations ---
sales_ts = sales.sort_values('Order Date').reset_index(drop=True)
sales_ts['Sales 3-Period MA'] = sales_ts['Sales'].rolling(window=3).mean()
sales_ts['Sales 3-Period Sum'] = sales_ts['Sales'].rolling(window=3).sum()
sales_ts['Sales Change'] = sales_ts['Sales'].diff()
sales_ts['Sales Pct Change'] = sales_ts['Sales'].pct_change()
sales_ts['Sales Lag 1'] = sales_ts['Sales'].shift(1)

print('Time-ordered sales with rolling and shift features:')
print(sales_ts[['OrderID', 'Order Date', 'Sales', 'Sales 3-Period MA', 'Sales Change', 'Sales Pct Change', 'Sales Lag 1']])

Time-ordered sales with rolling and shift features:
   OrderID Order Date  Sales  Sales 3-Period MA  Sales Change  \
0      101 2024-01-10   1200                NaN           NaN   
1      102 2024-01-15    800                NaN        -400.0   
2      103 2024-02-01   1250        1083.333333         450.0   
3      104 2024-02-18    900         983.333333        -350.0   
4      105 2024-03-05    820         990.000000         -80.0   
5      106 2024-03-10   1300        1006.666667         480.0   
6      107 2024-03-12    950        1023.333333        -350.0   
7      108 2024-04-01    780        1010.000000        -170.0   

   Sales Pct Change  Sales Lag 1  
0               NaN          NaN  
1         -0.333333       1200.0  
2          0.562500        800.0  
3         -0.280000       1250.0  
4         -0.088889        900.0  
5          0.585366        820.0  
6         -0.269231       1300.0  
7         -0.178947        950.0  


---
## 5. Time Series Basics with DateIndex and Resampling

Time series data is central to forecasting, seasonality analysis, and business reporting.

### Time Series Tools

| Method | Purpose | Example |
|--------|---------|---------|
| `pd.to_datetime()` | Convert text to datetime | `pd.to_datetime(df['Date'])` |
| `.set_index()` | Set DateTime index | `df.set_index('Date')` |
| `.resample()` | Aggregate by period | `df.resample('M').sum()` |
| `.asfreq()` | Change frequency | `df.asfreq('D')` |
| `.shift()` | Lag features | `df['Sales'].shift(1)` |

> ⚠️ **Special Case — resample only works with DateTimeIndex**: Convert the date column to datetime and set it as the index first.

In [8]:
# --- Time series resampling ---
revenue = pd.DataFrame({
    'Date': pd.to_datetime(['2024-01-05', '2024-01-12', '2024-01-19', '2024-02-02', '2024-02-09', '2024-03-01', '2024-03-08', '2024-03-15']),
    'Revenue': [1500, 1800, 1700, 2000, 1900, 2100, 2200, 2050],
    'Costs': [900, 950, 920, 980, 970, 1020, 1050, 1010]
})

revenue = revenue.set_index('Date')
print('Daily revenue data:')
print(revenue)
print()

monthly = revenue.resample('ME').sum()
print('Monthly revenue and cost totals:')
print(monthly)
print()

weekly = revenue.resample('W').mean()
print('Weekly average revenue and cost:')
print(weekly)
print()

revenue['Profit'] = revenue['Revenue'] - revenue['Costs']
print('Profit with monthly resample:')
print(revenue['Profit'].resample('ME').sum())
print()

print('Add monthly frequency and forward-fill missing dates:')
print(revenue.asfreq('D').ffill().head(10))

Daily revenue data:
            Revenue  Costs
Date                      
2024-01-05     1500    900
2024-01-12     1800    950
2024-01-19     1700    920
2024-02-02     2000    980
2024-02-09     1900    970
2024-03-01     2100   1020
2024-03-08     2200   1050
2024-03-15     2050   1010

Monthly revenue and cost totals:
            Revenue  Costs
Date                      
2024-01-31     5000   2770
2024-02-29     3900   1950
2024-03-31     6350   3080

Weekly average revenue and cost:
            Revenue   Costs
Date                       
2024-01-07   1500.0   900.0
2024-01-14   1800.0   950.0
2024-01-21   1700.0   920.0
2024-01-28      NaN     NaN
2024-02-04   2000.0   980.0
2024-02-11   1900.0   970.0
2024-02-18      NaN     NaN
2024-02-25      NaN     NaN
2024-03-03   2100.0  1020.0
2024-03-10   2200.0  1050.0
2024-03-17   2050.0  1010.0

Profit with monthly resample:
Date
2024-01-31    2230
2024-02-29    1950
2024-03-31    3270
Freq: ME, Name: Profit, dtype: int64

Add monthly 

---
## 6. Chaining and Method Pipelining

Chaining keeps pandas code readable and expressive. Use `.assign()`, `.query()`, and `.pipe()` for clean transformations.

### Chaining Best Practices

| Pattern | Purpose | Example |
|--------|---------|---------|
| `df.assign()` | Add new columns in chain | `df.assign(Profit=df['Sales']-df['Cost'])` |
| `.query()` | Filter in chain | `df.query('Sales > 1000')` |
| `.pipe()` | Apply custom function | `df.pipe(my_transform)` |
| `reset_index()` | Restore index after grouping | `...reset_index()` |

> **Data Science relevance**: Chaining helps keep data preparation pipelines easy to read and validate.

In [9]:
# --- Pandas chaining example ---
cleaned = (sales
    .query('Sales >= 800')
    .assign(RevenuePerQuantity=lambda df: df['Sales'] / df['Quantity'])
    .sort_values(['State', 'RevenuePerQuantity'], ascending=[True, False])
    .reset_index(drop=True)
)

print('Chained transformation result:')
print(cleaned[['OrderID', 'State', 'Sales', 'Quantity', 'RevenuePerQuantity']])

Chained transformation result:
   OrderID State  Sales  Quantity  RevenuePerQuantity
0      103    CA   1250         1              1250.0
1      101    CA   1200         1              1200.0
2      107    CA    950         1               950.0
3      102    NY    800         2               400.0
4      106    NY   1300         4               325.0
5      105    TX    820         2               410.0
6      104    TX    900         3               300.0


---
## 🧪 Practice Exercises

1. Use `.query()` to select orders from `sales` with `Quantity >= 2` and `Profit > 150`.
2. Find the top 2 customers by total sales using `.groupby()` and `.nlargest()`.
3. Create a new column `Risk` with `High` if `Profit < 120` else `Low`. Use `np.where()`.
4. Build a 2-period rolling average of `Revenue` from the time series data and compare it to the raw values.
5. Resample the `revenue` dataset by week and calculate the weekly profit sum.

---
## 📝 Session 10 Summary

| Topic | Key Idea |
|-------|----------|
| Advanced filtering | Use `.query()`, `.isin()`, `.between()`, and text filters for precise subsets |
| Sorting & ranking | Use `.sort_values()`, `.nlargest()`, `.rank()` to identify leaders and outliers |
| Conditional creation | Use `np.where()`, `.where()`, `.mask()`, `.clip()` to shape features |
| Window functions | `.rolling()`, `.shift()`, `.pct_change()` capture trends and momentum |
| Time series | Resample by period and use DateTimeIndex for frequency-based analytics |
| Chaining | Keep transformations readable and maintainable with `.assign()`, `.query()`, `.pipe()` |

### Key Takeaways
- Filtering and sorting are the first step to understanding any dataset.
- Conditional transformations are the foundation of feature engineering.
- Window calculations are essential for financial, marketing, and time-based analysis.
- Resampling converts raw time series into business-friendly summaries.
- Method chaining leads to cleaner and more predictable data pipelines.

### Next Session
**Session 11**: Data Visualization with Matplotlib and Seaborn — turning cleaned data into insight.